# Progetto agenti AI per Data

## Librerie

In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time

from dotenv import load_dotenv
from typing import Literal

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langgraph.checkpoint.memory import MemorySaver

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers.multi_query import MultiQueryRetriever 

In [5]:
load_dotenv(dotenv_path=".env")

True

## Creazione dei dataset

In [7]:
np.random.seed(42)

# Generazione date (ultimi 30 giorni)
dates = pd.date_range(end=datetime.today(), periods=24*30, freq='h')


# Dataset Produzione Solare (con qualche dato sporco/nullo)

production = np.maximum(0, np.sin((dates.hour.to_numpy() - 6) * np.pi / 12) * 5 + np.random.normal(0, 0.5, len(dates)))
production[dates.hour < 6] = 0 # Notte
production[dates.hour > 20] = 0 # Notte
df_prod = pd.DataFrame({'timestamp': dates, 'kwh_produced': production})


# Introduciamo anomalie (dati sporchi)

df_prod.loc[100:105, 'kwh_produced'] = np.nan # Sensore rotto
df_prod.loc[200, 'kwh_produced'] = -5 # Errore sensore (valore negativo impossibile)


# Dataset Consumi Utente

consumption = np.random.normal(1.5, 0.5, len(dates)) # Consumo base
consumption[dates.hour.isin([19, 20, 21])] += 2 # Picco serale
df_cons = pd.DataFrame({'timestamp': dates, 'kwh_consumed': consumption})


# Salvataggio

df_prod.to_csv("solar_production_raw.csv", index=False)
df_cons.to_csv("household_consumption.csv", index=False)

print("Dataset generati: 'solar_production_raw.csv' e 'household_consumption.csv'")

Dataset generati: 'solar_production_raw.csv' e 'household_consumption.csv'


## Tool 1 - caricamento e pulizia dati

In [9]:
@tool
def load_and_clean_data() -> str:
    """
    Carica i CSV di produzione solare e consumo domestico, pulisce i dati sporchi
    (valori NaN dovuti a sensori rotti, valori negativi impossibili) e li tiene
    in memoria (variabili globali df_prod_clean / df_cons_clean) invece di
    rileggerli da disco ad ogni chiamata. Salva comunque una copia su disco
    ('solar_production_clean.csv' e 'household_consumption_clean.csv') per
    ispezione manuale, ma gli altri tool usano get_clean_data() per leggere
    dalla cache in memoria, molto più efficiente.

    Usa questo tool come primo passo prima di qualsiasi analisi sui dati.

    Returns:
        Un riepilogo testuale delle anomalie trovate e corrette.
    """
    global df_prod_clean, df_cons_clean
    report = []

    # --- Produzione solare ---
    df_prod = pd.read_csv("solar_production_raw.csv", parse_dates=["timestamp"])
    df_prod = df_prod.sort_values("timestamp").reset_index(drop=True)

    n_negativi = (df_prod["kwh_produced"] < 0).sum()
    df_prod["kwh_produced"] = df_prod["kwh_produced"].clip(lower=0)

    n_nan = df_prod["kwh_produced"].isna().sum()
    df_prod["kwh_produced"] = df_prod["kwh_produced"].interpolate(method="linear")
    # eventuali NaN residui a inizio/fine serie
    df_prod["kwh_produced"] = df_prod["kwh_produced"].fillna(0)

    df_prod.to_csv("solar_production_clean.csv", index=False)
    report.append(f"Produzione: corretti {n_negativi} valori negativi impossibili, "
                  f"interpolati {n_nan} valori mancanti (sensore guasto).")

    # --- Consumo domestico ---
    df_cons = pd.read_csv("household_consumption.csv", parse_dates=["timestamp"])
    df_cons = df_cons.sort_values("timestamp").reset_index(drop=True)

    n_negativi_cons = (df_cons["kwh_consumed"] < 0).sum()
    df_cons["kwh_consumed"] = df_cons["kwh_consumed"].clip(lower=0)

    n_nan_cons = df_cons["kwh_consumed"].isna().sum()
    df_cons["kwh_consumed"] = df_cons["kwh_consumed"].interpolate(method="linear").fillna(0)

    df_cons.to_csv("household_consumption_clean.csv", index=False)
    report.append(f"Consumo: corretti {n_negativi_cons} valori negativi, "
                  f"interpolati {n_nan_cons} valori mancanti.")

    # Aggiorniamo la cache in memoria: gli altri tool la useranno invece di
    # rileggere i CSV da disco ad ogni chiamata.
    df_prod_clean = df_prod
    df_cons_clean = df_cons

    return " ".join(report)


def get_clean_data():
    """
    Ritorna (df_prod_clean, df_cons_clean) dalla cache in memoria. Se
    load_and_clean_data() non è ancora stato eseguito in questa sessione,
    lo esegue automaticamente una volta sola.
    """
    global df_prod_clean, df_cons_clean
    if df_prod_clean is None or df_cons_clean is None:
        load_and_clean_data.invoke({})
    return df_prod_clean, df_cons_clean


# Variabili globali usate come cache: valgono None finché load_and_clean_data
# non viene eseguito almeno una volta.
df_prod_clean = None
df_cons_clean = None

**Nota su performance (revisione post-consegna):** nella prima versione, ogni tool
successivo (`calculate_production`, `calculate_consumption`, ecc.) rileggeva da zero i CSV puliti
da disco tramite `pd.read_csv` ad ogni singola chiamata — anche più volte nello stesso turno di
conversazione. Ora `load_and_clean_data` popola due variabili globali (`df_prod_clean`,
`df_cons_clean`) usate come cache in memoria da tutti gli altri tool tramite `get_clean_data()`,
evitando letture e parsing ripetuti e inutili. In app.py la stessa idea è realizzata con
`@st.cache_data`, l'equivalente "streamlit-native" di questa cache.

In [11]:
result = load_and_clean_data.invoke({})
print(result)

Produzione: corretti 1 valori negativi impossibili, interpolati 6 valori mancanti (sensore guasto). Consumo: corretti 0 valori negativi, interpolati 0 valori mancanti.


## Tool 2 - Energia prodotta

In [13]:
@tool
def calculate_production(period: Literal["hour", "day", "week", "total"] = "total") -> str:
    """
    Calcola l'energia prodotta dai pannelli solari dell'azienda, aggregata
    per il periodo richiesto. Usa i dati puliti (esegui load_and_clean_data
    prima se non l'hai già fatto).

    Args:
        period: livello di aggregazione.
            "total" = somma totale sull'intero dataset (ultimi 30 giorni)
            "day"   = somma per ogni giorno (mostra andamento giornaliero)
            "week"  = somma per settimana
            "hour"  = media di produzione per ogni ora del giorno (0-23),
                      utile per capire in che fascia oraria si produce di più

    Returns:
        Un riepilogo testuale dei kWh prodotti secondo l'aggregazione scelta.
    """
    df, _ = get_clean_data()  # letto dalla cache in memoria, non da disco

    if period == "total":
        totale = df["kwh_produced"].sum()
        return f"Produzione totale negli ultimi 30 giorni: {totale:.2f} kWh."

    elif period == "day":
        daily = df.set_index("timestamp").resample("D")["kwh_produced"].sum()
        righe = [f"{d.strftime('%d/%m')}: {v:.2f} kWh" for d, v in daily.items()]
        return "Produzione giornaliera:\n" + "\n".join(righe)

    elif period == "week":
        weekly = df.set_index("timestamp").resample("W")["kwh_produced"].sum()
        righe = [f"Settimana del {d.strftime('%d/%m')}: {v:.2f} kWh" for d, v in weekly.items()]
        return "Produzione settimanale:\n" + "\n".join(righe)

    elif period == "hour":
        hourly = df.groupby(df["timestamp"].dt.hour)["kwh_produced"].mean()
        righe = [f"ore {h}:00 -> {v:.2f} kWh medi" for h, v in hourly.items()]
        return "Produzione media per ora del giorno:\n" + "\n".join(righe)

## Tool 3 - Energia consumata

In [15]:
@tool
def calculate_consumption(period: Literal["hour", "day", "week", "total"] = "total") -> str:
    """
    Calcola l'energia consumata dalla famiglia, aggregata per il periodo richiesto.
    Usa i dati puliti (esegui load_and_clean_data prima se non l'hai già fatto).

    Args:
        period: livello di aggregazione ("total", "day", "week", "hour" -
            stesso significato di calculate_production).

    Returns:
        Un riepilogo testuale dei kWh consumati secondo l'aggregazione scelta.
    """
    _, df = get_clean_data()  # letto dalla cache in memoria, non da disco

    if period == "total":
        totale = df["kwh_consumed"].sum()
        return f"Consumo totale negli ultimi 30 giorni: {totale:.2f} kWh."

    elif period == "day":
        daily = df.set_index("timestamp").resample("D")["kwh_consumed"].sum()
        righe = [f"{d.strftime('%d/%m')}: {v:.2f} kWh" for d, v in daily.items()]
        return "Consumo giornaliero:\n" + "\n".join(righe)

    elif period == "week":
        weekly = df.set_index("timestamp").resample("W")["kwh_consumed"].sum()
        righe = [f"Settimana del {d.strftime('%d/%m')}: {v:.2f} kWh" for d, v in weekly.items()]
        return "Consumo settimanale:\n" + "\n".join(righe)

    elif period == "hour":
        hourly = df.groupby(df["timestamp"].dt.hour)["kwh_consumed"].mean()
        righe = [f"ore {h}:00 -> {v:.2f} kWh medi" for h, v in hourly.items()]
        return "Consumo medio per ora del giorno:\n" + "\n".join(righe)

In [16]:
print(calculate_production.invoke({"period": "week"}))
print(calculate_consumption.invoke({"period": "hour"}))

Produzione settimanale:
Settimana del 02/08: 109.93 kWh
Settimana del 09/08: 273.25 kWh
Settimana del 16/08: 269.07 kWh
Settimana del 23/08: 262.87 kWh
Settimana del 30/08: 230.19 kWh
Consumo medio per ora del giorno:
ore 0:00 -> 1.65 kWh medi
ore 1:00 -> 1.34 kWh medi
ore 2:00 -> 1.42 kWh medi
ore 3:00 -> 1.51 kWh medi
ore 4:00 -> 1.47 kWh medi
ore 5:00 -> 1.53 kWh medi
ore 6:00 -> 1.67 kWh medi
ore 7:00 -> 1.50 kWh medi
ore 8:00 -> 1.51 kWh medi
ore 9:00 -> 1.63 kWh medi
ore 10:00 -> 1.63 kWh medi
ore 11:00 -> 1.47 kWh medi
ore 12:00 -> 1.52 kWh medi
ore 13:00 -> 1.64 kWh medi
ore 14:00 -> 1.60 kWh medi
ore 15:00 -> 1.42 kWh medi
ore 16:00 -> 1.63 kWh medi
ore 17:00 -> 1.50 kWh medi
ore 18:00 -> 1.55 kWh medi
ore 19:00 -> 3.46 kWh medi
ore 20:00 -> 3.56 kWh medi
ore 21:00 -> 3.71 kWh medi
ore 22:00 -> 1.50 kWh medi
ore 23:00 -> 1.66 kWh medi


- Produzione settimanale: le due settimane agli estremi del range di 30 giorni mostrano valori più bassi perché contengono meno giorni — coerente.
- Consumo orario: si vede perfettamente il picco serale programmato nello script, tra le 19 e le 21. Questo l'agente dovrà saperlo spiegare all'utente ("consumi di più la sera perché...").

I prossimi due tool sono più complessi dei precedenti: non si limitano ad aggregare, ma incrociano i due dataset per dare all'agente materiale su cui basare i consigli.

## Tool 4 - Confronto consumi

In [20]:
@tool
def compare_production_consumption(period: Literal["day", "week"] = "day", last_n: int = None) -> str:
    """
    Confronta produzione ed energia consumata nello stesso periodo, evidenziando
    surplus o deficit. Usa i dati puliti.

    Args:
        period: "day" per confronto giornaliero, "week" per confronto settimanale.
        last_n: se specificato, limita l'output agli ultimi N periodi (es. last_n=2
            con period="week" per le ultime 2 settimane) e aggiunge un totale
            aggregato già calcolato, così non serve sommare i valori a mano.

    Returns:
        Un riepilogo testuale con produzione, consumo e saldo per ogni periodo,
        più un totale aggregato se last_n è specificato.
    """
    df_prod, df_cons = get_clean_data()  # letti dalla cache in memoria, non da disco

    freq = "D" if period == "day" else "W"
    prod_agg = df_prod.set_index("timestamp").resample(freq)["kwh_produced"].sum()
    cons_agg = df_cons.set_index("timestamp").resample(freq)["kwh_consumed"].sum()

    df = pd.DataFrame({"prodotto": prod_agg, "consumato": cons_agg})
    df["saldo"] = df["prodotto"] - df["consumato"]

    if last_n:
        df = df.tail(last_n)

    label = "Giorno" if period == "day" else "Settimana del"
    righe = []
    for ts, row in df.iterrows():
        stato = "surplus" if row["saldo"] >= 0 else "deficit"
        righe.append(
            f"{label} {ts.strftime('%d/%m')}: prodotti {row['prodotto']:.2f} kWh, "
            f"consumati {row['consumato']:.2f} kWh, saldo {row['saldo']:+.2f} kWh ({stato})"
        )
    output = "\n".join(righe)

    if last_n and last_n > 1:
        tot_prod, tot_cons = df["prodotto"].sum(), df["consumato"].sum()
        tot_saldo = tot_prod - tot_cons
        stato = "surplus" if tot_saldo >= 0 else "deficit"
        output += (f"\n\nTotale sulle ultime {last_n} {period}e: prodotti {tot_prod:.2f} kWh, "
                   f"consumati {tot_cons:.2f} kWh, saldo {tot_saldo:+.2f} kWh ({stato})")

    return output

## Tool 5 - Individua anomalie
Questa funzione non deve rilevare gli stessi errori che vengono corretti da load_and_clean_data. Deve invece cercare anomalie comportamentali, quelle utili per i consigli, come "giorno con surplus enorme non sfruttato".

NB: I giorni con meno di 24 ore sono stati rimossi, il primo e l'ultimo, perché non sono rappresentativi dei consumi.

In [22]:
@tool
def detect_anomalies() -> str:
    """
    Individua situazioni particolari nei dati puliti, utili per generare consigli:
    giorni con surplus elevato di energia prodotta e non consumata (occasione
    sprecata) e giorni con consumo anomalmente alto rispetto alla media.
    I giorni parziali (inizio/fine dataset con meno di 24 ore registrate)
    vengono esclusi per evitare falsi positivi. Usa i dati puliti.

    Returns:
        Un riepilogo testuale delle situazioni degne di nota trovate.
    """
    df_prod, df_cons = get_clean_data()  # letti dalla cache in memoria, non da disco

    prod_daily = df_prod.set_index("timestamp").resample("D")["kwh_produced"].sum()
    cons_daily = df_cons.set_index("timestamp").resample("D")["kwh_consumed"].sum()

    # Esclude giorni con meno di 24 ore registrate (primo e ultimo)
    ore_per_giorno = df_cons.set_index("timestamp").resample("D")["kwh_consumed"].count()
    giorni_completi = ore_per_giorno[ore_per_giorno == 24].index

    prod_daily = prod_daily.loc[giorni_completi]
    cons_daily = cons_daily.loc[giorni_completi]

    saldo = prod_daily - cons_daily
    soglia_surplus = saldo.mean() + saldo.std()
    media_cons = cons_daily.mean()
    soglia_consumo_alto = media_cons + cons_daily.std()

    note = []
    for ts in saldo.index:
        if saldo[ts] > 0 and saldo[ts] > soglia_surplus:
            note.append(f"{ts.strftime('%d/%m')}: surplus elevato di {saldo[ts]:.2f} kWh "
                     f"non sfruttato (energia prodotta ma non consumata).")
        if cons_daily[ts] > soglia_consumo_alto:
            note.append(f"{ts.strftime('%d/%m')}: consumo insolitamente alto "
                     f"({cons_daily[ts]:.2f} kWh, media {media_cons:.2f} kWh).")
  
    if not note:
        return "Nessuna anomalia particolare rilevata nei dati puliti."
    return "Situazioni rilevanti trovate:\n" + "\n".join(note)

In [23]:
# Test per tool 4 e 5
print(compare_production_consumption.invoke({"period": "day"}))
print(detect_anomalies.invoke({}))

Giorno 30/07: prodotti 0.25 kWh, consumati 13.99 kWh, saldo -13.74 kWh (deficit)
Giorno 31/07: prodotti 34.43 kWh, consumati 39.94 kWh, saldo -5.51 kWh (deficit)
Giorno 01/08: prodotti 36.16 kWh, consumati 45.57 kWh, saldo -9.41 kWh (deficit)
Giorno 02/08: prodotti 39.09 kWh, consumati 41.95 kWh, saldo -2.86 kWh (deficit)
Giorno 03/08: prodotti 37.58 kWh, consumati 43.62 kWh, saldo -6.04 kWh (deficit)
Giorno 04/08: prodotti 39.16 kWh, consumati 42.66 kWh, saldo -3.51 kWh (deficit)
Giorno 05/08: prodotti 37.96 kWh, consumati 44.92 kWh, saldo -6.96 kWh (deficit)
Giorno 06/08: prodotti 41.00 kWh, consumati 47.06 kWh, saldo -6.06 kWh (deficit)
Giorno 07/08: prodotti 37.15 kWh, consumati 46.42 kWh, saldo -9.27 kWh (deficit)
Giorno 08/08: prodotti 42.09 kWh, consumati 40.40 kWh, saldo +1.69 kWh (surplus)
Giorno 09/08: prodotti 38.32 kWh, consumati 45.35 kWh, saldo -7.02 kWh (deficit)
Giorno 10/08: prodotti 36.82 kWh, consumati 39.24 kWh, saldo -2.42 kWh (deficit)
Giorno 11/08: prodotti 38.63

## Tool 6 - Grafico
Ultimo tool per i dati, prima di passare al reasoning loop. Produce un file immagine che l'agente mostra nella risposta. Il tool restituisce il path del PNG salvato, non i dati grezzi — sarà poi Streamlit a leggerlo e visualizzarlo nella chat.

In [25]:
@tool
def generate_chart(
    metric: Literal["production", "consumption", "comparison"] = "comparison",
    period: Literal["day", "week"] = "day",
    weeks_back: int = 0
) -> str:
    """
    Genera un grafico e lo salva come PNG in 'charts/'. Usa questo tool quando
    l'utente chiede di vedere un grafico, un andamento o un confronto visivo.

    Args:
        metric: "production", "consumption" o "comparison".
        period: "day" per andamento a 24h, "week" per andamento settimanale (7 giorni).
        weeks_back: solo se period="week". Quante settimane indietro rispetto
            alla più recente disponibile: 0 = ultima settimana, 1 = quella
            precedente, ecc. Usa questo per rispondere a richieste tipo
            "la settimana prima" o "due settimane fa".

    Returns:
        Il percorso del file PNG generato.
    """
    os.makedirs("charts", exist_ok=True)
    df_prod, df_cons = get_clean_data()  # letti dalla cache in memoria, non da disco

    fig, ax = plt.subplots(figsize=(8, 4))

    if period == "day":
        prod_plot = df_prod.groupby(df_prod["timestamp"].dt.hour)["kwh_produced"].mean()
        cons_plot = df_cons.groupby(df_cons["timestamp"].dt.hour)["kwh_consumed"].mean()
        xlabel = "Ora del giorno"
    else:
        prod_daily = df_prod.set_index("timestamp").resample("D")["kwh_produced"].sum()
        cons_daily = df_cons.set_index("timestamp").resample("D")["kwh_consumed"].sum()

        end = len(prod_daily) - weeks_back * 7
        start = max(0, end - 7)
        prod_plot = prod_daily.iloc[start:end]
        cons_plot = cons_daily.iloc[start:end]
        prod_plot.index = prod_plot.index.strftime("%d/%m")
        cons_plot.index = cons_plot.index.strftime("%d/%m")
        xlabel = "Giorno"

    if metric in ("production", "comparison"):
        ax.plot(prod_plot.index, prod_plot.values, label="Produzione (kWh)", color="#f4a261", marker="o")
    if metric in ("consumption", "comparison"):
        ax.plot(cons_plot.index, cons_plot.values, label="Consumo (kWh)", color="#2a9d8f", marker="o")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("kWh")
    ax.set_title(f"{metric.capitalize()} - andamento per {period}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()

    filename = f"charts/{metric}_{period}_back{weeks_back}.png"
    fig.savefig(filename)
    plt.close(fig)
    return filename

- Salva i grafici in una sottocartella charts/ invece che nella root, per tenere ordinato il progetto.
- plt.close(fig) dopo il salvataggio: essenziale quando il tool viene chiamato più volte in una sessione (es. l'agente genera più grafici in una conversazione) — altrimenti matplotlib accumula figure in memoria.

In [27]:
# test
path = generate_chart.invoke({"metric": "comparison", "period": "day"})
print(path)

charts/comparison_day_back0.png


## Tool 7 - RAG (base di conoscenza)
I tool precedenti rispondono solo a domande sui **numeri** (produzione, consumo, saldo). Molte
domande degli utenti sono invece **concettuali** ("perché la bolletta è alta anche con i pannelli
solari?", "cos'è una comunità energetica?"), e per queste l'agente non deve inventare una risposta
ma appoggiarsi a una base di conoscenza scritta da EcoGrid.

Costruiamo quindi un piccolo indice vettoriale (RAG) su due documenti di esempio (una guida al
risparmio energetico e una FAQ su tariffe/CER), usando embeddings locali (sentence-transformers,
nessuna API key aggiuntiva) e FAISS come vector store.

**Nota (revisione post-consegna, seconda iterazione):** il retrieval è a tre stadi:
1. `MultiQueryRetriever` usa l'LLM per generare 3 riformulazioni della domanda (sinonimi, termini
   tecnici diversi), recupera i chunk più simili per ciascuna e unisce i risultati deduplicati —
   risolve il problema del vocabolario semantico (es. "batteria" vs "sistema di accumulo").
2. FAISS recupera, per ciascuna riformulazione, gli 8 chunk più simili per similarità tra embedding.
3. Un **cross-encoder** (`CrossEncoderReranker`, locale) rilegge l'unione dei candidati insieme
   alla domanda originale e tiene solo i 5 migliori.

In LangChain la pipeline si costruisce componendo `MultiQueryRetriever` (che avvolge il retriever
FAISS) dentro un `ContextualCompressionRetriever` (che applica il reranking sul risultato).

In [29]:
os.makedirs("knowledge_base", exist_ok=True)

# In produzione questi file arrivano già pronti nella cartella knowledge_base/

guida = """# Guida al risparmio energetico per famiglie con fotovoltaico

## Come funziona l'autoconsumo
Quando un impianto fotovoltaico produce energia, questa viene prima usata per coprire i consumi
della casa in quel preciso momento (autoconsumo). Solo l'energia prodotta e non consumata in tempo
reale viene immessa in rete (o condivisa con la comunità energetica). Spostare i consumi nelle ore
di massima produzione (tipicamente tra le 10:00 e le 16:00) riduce la quota acquistata dalla rete.

## Perché la bolletta può essere alta anche con i pannelli solari
Un impianto fotovoltaico produce energia solo durante le ore di luce, mentre molte famiglie
concentrano i consumi nella fascia serale, quando la produzione è nulla. In questi casi l'energia
serale viene acquistata dalla rete a prezzo pieno, anche se durante il giorno l'impianto ha
prodotto un surplus non utilizzato. La soluzione più efficace è spostare i consumi nelle ore
diurne, oppure valutare un sistema di accumulo (batteria).

## Sistemi di accumulo (batterie domestiche)
Una batteria domestica immagazzina l'energia prodotta in eccesso durante il giorno e la rende
disponibile la sera. Ha senso soprattutto per famiglie con surplus diurni elevati e ricorrenti,
ma consumi concentrati nella fascia serale.
"""

faq = """# FAQ - Tariffe energetiche e Comunità Energetiche Rinnovabili (CER)

## Cos'è una Comunità Energetica Rinnovabile (CER)?
Una CER è un gruppo di soggetti che si associano per produrre, condividere e consumare energia
rinnovabile prodotta localmente, riducendo la dipendenza dalla rete elettrica tradizionale.

## Come si guadagna vendendo energia in eccesso?
Un produttore locale che genera più energia di quanta ne consuma può immettere il surplus nella
rete della comunità. Chi consuma nello stesso momento in cui il surplus viene immesso beneficia
di un incentivo economico condiviso.

## Cosa fare in caso di deficit ricorrente
Spostare gli elettrodomestici ad alto consumo nelle ore diurne, valutare una batteria di accumulo,
oppure verificare con EcoGrid eventuali tariffe della comunità più vantaggiose per la sera.
"""

with open("knowledge_base/guida_risparmio_energetico.md", "w", encoding="utf-8") as f:
    f.write(guida)
with open("knowledge_base/faq_tariffe_cer.md", "w", encoding="utf-8") as f:
    f.write(faq)

print("Base di conoscenza creata in knowledge_base/")

Base di conoscenza creata in knowledge_base/


In [30]:
# Cache del retriever: costruito una sola volta, non ad ogni domanda.
#
# Pipeline di retrieval a tre stadi (revisione post-consegna):
# 1) MultiQueryRetriever usa l'LLM (Groq) per generare 3 riformulazioni della
#    domanda (sinonimi, termini tecnici diversi, altri modi di dire la stessa
#    cosa) e recupera i chunk più simili per ciascuna, poi unisce i risultati
#    eliminando i duplicati. Risolve il problema del "vocabolario semantico":
#    se l'utente scrive "conviene una batteria?" ma il documento parla di
#    "sistemi di accumulo", più riformulazioni aumentano le probabilità di
#    intercettare il passaggio giusto. Riduce anche i token più avanti nella
#    pipeline: i duplicati (stesso chunk da più riformulazioni) contano una
#    sola volta.
# 2) FAISS recupera, per ciascuna riformulazione, gli 8 chunk più simili per
#    similarità tra embedding.
# 3) Un cross-encoder (CrossEncoderReranker) rilegge l'unione dei chunk
#    recuperati insieme alla domanda ORIGINALE e li riordina per pertinenza
#    logica effettiva. Solo i 5 migliori (top_n) vengono passati all'agente.
# Embeddings e cross-encoder girano in locale; le riformulazioni usano invece
# l'LLM Groq, quindi ogni domanda concettuale genera una chiamata extra a
# Groq (oltre a quella per la risposta finale) — da tenere presente sul
# fronte del rate limit del free tier (gestito da invoke_agent_with_retry).
_retriever = None

# Prompt italiano per le riformulazioni: quello di default di
# MultiQueryRetriever è in inglese, ma domande e documenti qui sono italiani.
MULTI_QUERY_PROMPT_IT = PromptTemplate(
    input_variables=["question"],
    template="""Sei un assistente che aiuta a migliorare la ricerca in una base di conoscenza
sull'energia domestica e il fotovoltaico. Genera 3 riformulazioni diverse della domanda
dell'utente qui sotto, usando sinonimi, termini tecnici alternativi o modi di dire diversi
per esprimere lo stesso concetto, così da recuperare documenti pertinenti anche se la domanda
originale usa un vocabolario diverso da quello dei documenti. Scrivi le 3 riformulazioni in
italiano, una per riga, senza numerarle e senza aggiungere altro testo.

Domanda originale: {question}""",
)


def _get_retriever():
    global _retriever
    if _retriever is None:
        loader = DirectoryLoader("knowledge_base", glob="**/*.md", loader_cls=TextLoader,
                                  loader_kwargs={"encoding": "utf-8"})
        documents = loader.load()

        # chunk_size aumentato (da 800 a 1200): con chunk troppo piccoli,
        # concetti articolati su più frasi rischiano di essere spezzati a
        # metà, perdendo il contesto necessario al cross-encoder.
        splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
        chunks = splitter.split_documents(documents)

        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = FAISS.from_documents(chunks, embeddings)

        # Stadio 1: recupero per similarità (k=8 per ciascuna riformulazione)
        base_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

        # Stadio 2: MultiQueryRetriever genera le riformulazioni con l'LLM
        # Groq e unisce i risultati (deduplicati) di tutte le varianti.
        query_llm = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0.3,
            api_key=os.getenv("GROQ_API_KEY"),
        )
        multi_query_retriever = MultiQueryRetriever.from_llm(
            retriever=base_retriever,
            llm=query_llm,
            prompt=MULTI_QUERY_PROMPT_IT,
            include_original=True,
        )

        # Stadio 3: reranking con cross-encoder locale, tiene solo i 5 migliori
        # rispetto alla domanda originale dell'utente.
        cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
        reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

        _retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=multi_query_retriever)
    return _retriever


@tool
def search_knowledge_base(query: str) -> str:
    """
    Cerca nella base di conoscenza di EcoGrid (guide sul risparmio energetico,
    FAQ su tariffe e comunità energetiche rinnovabili) informazioni utili per
    rispondere a domande concettuali, quando non si tratta di una domanda sui
    dati numerici (produzione/consumo/saldo) dell'utente.

    Esempi di domande adatte a questo tool: "Perché la bolletta è alta anche
    se ho i pannelli solari?", "Cos'è una comunità energetica rinnovabile?",
    "Conviene installare una batteria?".

    Args:
        query: la domanda o l'argomento da cercare nella base di conoscenza.

    Returns:
        I 5 passaggi più rilevanti (dopo riformulazioni multiple e reranking)
        trovati nei documenti, da usare come base per la risposta.
    """
    retriever = _get_retriever()
    risultati = retriever.invoke(query)

    if not risultati:
        return "Nessuna informazione pertinente trovata nella base di conoscenza."

    passaggi = [f"[Fonte: {os.path.basename(r.metadata.get('source', '?'))}]\n{r.page_content}" for r in risultati]
    return "\n\n---\n\n".join(passaggi)

In [31]:
# test
print(search_knowledge_base.invoke({"query": "perché la bolletta è alta anche con i pannelli solari?"}))

[Fonte: guida_risparmio_energetico.md]
# Guida al risparmio energetico per famiglie con fotovoltaico

## Come funziona l'autoconsumo
Quando un impianto fotovoltaico produce energia, questa viene prima usata per coprire i consumi
della casa in quel preciso momento (autoconsumo). Solo l'energia prodotta e non consumata in tempo
reale viene immessa in rete (o condivisa con la comunità energetica). Spostare i consumi nelle ore
di massima produzione (tipicamente tra le 10:00 e le 16:00) riduce la quota acquistata dalla rete.

## Perché la bolletta può essere alta anche con i pannelli solari
Un impianto fotovoltaico produce energia solo durante le ore di luce, mentre molte famiglie
concentrano i consumi nella fascia serale, quando la produzione è nulla. In questi casi l'energia
serale viene acquistata dalla rete a prezzo pieno, anche se durante il giorno l'impianto ha
prodotto un surplus non utilizzato. La soluzione più efficace è spostare i consumi nelle ore
diurne, oppure valutare un siste

## Costruzione dell'agente

In [33]:
system_prompt = """Sei EcoBot, un assistente virtuale specializzato in analisi energetica residenziale.
Aiuti l'utente a capire i dati di produzione fotovoltaica e di consumo domestico della sua abitazione.

Il tuo tono è professionale ma cordiale: chiaro, preciso, senza gergo tecnico non necessario.
Ti rivolgi all'utente con "tu", in modo diretto ma sempre rispettoso.

Regole importanti:
- Usa SEMPRE i tool a disposizione per rispondere a domande su dati, numeri o statistiche.
  Non inventare mai valori: se non hai un tool adatto per rispondere con certezza, dillo esplicitamente.
- Se i dati non sono ancora stati puliti (prima interazione), usa load_and_clean_data come primo passo.
- Se l'utente chiede un grafico, un andamento, una curva o un confronto visivo, usa generate_chart
  e menziona nella risposta che il grafico è disponibile.
- Se l'utente fa una domanda di follow-up che si riferisce a un grafico mostrato in un turno
  precedente (es. "e la settimana prima?", "quella prima ancora?", "confrontala con l'altra"),
  continua a generare il grafico corrispondente con generate_chart, anche se il follow-up non
  ripete esplicitamente parole come "grafico" o "andamento".
- IMPORTANTE: non scrivere mai frasi come "ecco il grafico" o "il grafico è disponibile" se non hai
  effettivamente chiamato generate_chart in questo stesso turno. Se non hai generato un grafico,
  non parlarne come se esistesse.
- Non citare mai all'utente i nomi tecnici dei tool (es. "detect_anomalies", "search_knowledge_base",
  "generate_chart"): sono strumenti interni che tu usi per rispondere, non qualcosa che l'utente può
  eseguire. Se vuoi proporre un approfondimento, descrivilo in linguaggio naturale
  (es. "posso controllare se ci sono giorni con surplus significativo" invece di
  "posso eseguire detect_anomalies").
- Se descrivi i colori di un grafico generato con generate_chart, usa sempre e solo questi (sono fissi
  nel codice, non indovinarli): produzione = arancione, consumo = verde acqua (teal). Non usare mai
  altri colori (es. blu, rosso) per descriverli.
- Quando riporti numeri, arrotonda a 2 decimali e specifica sempre l'unità (kWh).
- Se noti surplus o deficit energetici rilevanti, offri un breve consiglio pratico
  (es. spostare l'uso di elettrodomestici nelle ore di massima produzione).
- Per domande concettuali (es. "perché la bolletta è alta anche con i pannelli solari?",
  "cos'è una comunità energetica?", "conviene una batteria?") usa il tool
  search_knowledge_base invece di rispondere solo dalla tua conoscenza generale.
- Rispondi sempre in italiano, in modo sintetico ma completo.
"""

In [34]:
load_dotenv(dotenv_path=".env")

llm = ChatGroq(
    model="openai/gpt-oss-120b",   # llama-3.3-70b-versatile è stato deprecato da Groq
    temperature=0.3,
    api_key=os.getenv("GROQ_API_KEY")
)

tools = [
    load_and_clean_data,
    calculate_production,
    calculate_consumption,
    compare_production_consumption,
    detect_anomalies,
    generate_chart,
    search_knowledge_base
]

checkpointer = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

**Perché queste scelte**

- MessagesPlaceholder("chat_history", optional=True): servirà in Streamlit per passare lo storico della conversazione — prepararlo ora evita di riscrivere il prompt dopo.
- verbose=True: fondamentale per il test nel notebook, perché mostra esattamente quale tool l'agente sceglie e con quali argomenti — quindi mostra il ragionamento (come richiesto dal brief).
- max_iterations=6: previene loop infiniti se il modello dovesse continuare a chiamare tool senza mai arrivare a una risposta finale — un limite di sicurezza richiesto implicitamente dal punto "gestione degli errori" del brief.

### Gestione degli errori — retry automatico su rate limit
Il free tier di Groq ha un limite di token al minuto (TPM). Se viene superato, l'API risponde con
un errore 429 che indica anche dopo quanti secondi/millisecondi riprovare (di solito meno di un
minuto). Invece di far fallire il test o la conversazione con uno stack trace, definiamo una
funzione che ritenta automaticamente con un breve backoff esponenziale (3s, 6s, 12s...), e la
usiamo al posto di agent.invoke() diretto nei test successivi (e nella versione Streamlit).

In [37]:
def invoke_agent_with_retry(agent, user_message: str, config: dict, max_retries: int = 3, base_delay_seconds: float = 3):
    """
    Chiama l'agente gestendo automaticamente gli errori di rate limit (429).
    Se la chiamata fallisce per un motivo diverso dal rate limit, l'eccezione
    viene rilanciata subito, senza ritentare inutilmente.

    Args:
        agent: l'agente LangGraph da invocare.
        user_message: il messaggio dell'utente per questo turno.
        config: la configurazione (thread_id) da passare ad agent.invoke.
        max_retries: numero massimo di tentativi aggiuntivi dopo il primo.
        base_delay_seconds: attesa base per il backoff esponenziale.

    Returns:
        Il risultato di agent.invoke, se uno dei tentativi va a buon fine.

    Raises:
        L'ultima eccezione incontrata, se anche l'ultimo tentativo fallisce.
    """
    for tentativo in range(max_retries + 1):
        try:
            return agent.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)
        except Exception as e:
            is_rate_limit = "rate_limit" in str(e).lower() or "429" in str(e)
            if not is_rate_limit or tentativo == max_retries:
                raise
            attesa = base_delay_seconds * (2 ** tentativo)
            print(f"⏳ Rate limit raggiunto, riprovo tra {attesa:.0f}s (tentativo {tentativo + 1}/{max_retries})...")
            time.sleep(attesa)

### Test A — reasoning multi-tool in un singolo turno (nessuna memoria richiesta)

In [39]:
config_a = {"configurable": {"thread_id": "test-multi-tool"}}

result = invoke_agent_with_retry(
    agent,
    "Confronta produzione e consumo delle ultime due settimane, segnalami eventuali anomalie e mostrami un grafico dell'andamento",
    config_a
)
print(result["messages"][-1].content)

**Confronto produzione‑consumo – ultime 2 settimane**

| Settimana | Produzione | Consumo | Saldo |
|-----------|------------|---------|-------|
| 23/08‑29/08 | **262.87 kWh** | **296.11 kWh** | **‑33.24 kWh** (deficit) |
| 30/08‑05/09 | **230.19 kWh** | **240.21 kWh** | **‑10.02 kWh** (deficit) |
| **Totale** | **493.06 kWh** | **536.33 kWh** | **‑43.27 kWh** (deficit) |

**Anomalie rilevate**

- **06/08, 07/08, 16/08, 28/08**: consumo insolitamente alto (≈ 47 kWh, sopra la media di 43.14 kWh).  
- **08/08, 25/08**: surplus di energia prodotta non sfruttata (1.69 kWh e 0.34 kWh).  

**Consiglio pratico**  
Sfrutta i picchi di produzione spostando l’uso di elettrodomestici ad alta potenza (lavatrice, asciugatrice, lavastoviglie) nelle ore di massima produzione (tipicamente tra le 10:00 e le 14:00). In questo modo ridurrai i deficit settimanali e potrai utilizzare l’energia che altrimenti verrebbe sprecata.

**Grafico**  
Ho generato un grafico settimanale che mostra produzione (arancio

La prossima cella è utile sia per debug sia come prova del reasoning loop. Mostra esattamente la sequenza di azioni dell'agente che hanno portato alla risposta precedente.

In [41]:
for msg in result["messages"]:
    tipo = type(msg).__name__
    if tipo == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Chiama tool: {tc['name']} con argomenti {tc['args']}")
    elif tipo == "ToolMessage":
        print(f"📥 Risultato da {msg.name}: {msg.content[:150]}...")
    elif tipo == "AIMessage" and msg.content:
        print(f"🤖 Risposta finale: {msg.content}")

🔧 Chiama tool: load_and_clean_data con argomenti {}
📥 Risultato da load_and_clean_data: Produzione: corretti 1 valori negativi impossibili, interpolati 6 valori mancanti (sensore guasto). Consumo: corretti 0 valori negativi, interpolati 0...
🔧 Chiama tool: compare_production_consumption con argomenti {'last_n': 2, 'period': 'week'}
📥 Risultato da compare_production_consumption: Settimana del 23/08: prodotti 262.87 kWh, consumati 296.11 kWh, saldo -33.24 kWh (deficit)
Settimana del 30/08: prodotti 230.19 kWh, consumati 240.21 ...
🔧 Chiama tool: detect_anomalies con argomenti {}
📥 Risultato da detect_anomalies: Situazioni rilevanti trovate:
06/08: consumo insolitamente alto (47.06 kWh, media 43.14 kWh).
07/08: consumo insolitamente alto (46.42 kWh, media 43.1...
🔧 Chiama tool: generate_chart con argomenti {'metric': 'comparison', 'period': 'week'}
📥 Risultato da generate_chart: charts/comparison_week_back0.png...
🤖 Risposta finale: **Confronto produzione‑consumo – ultime 2 settimane**

|

### Test B — memoria conversazionale multi-turno

In [43]:
config_b = {"configurable": {"thread_id": "test-memoria"}}

# Prima domanda
result1 = invoke_agent_with_retry(agent, "Qual è stata la produzione della scorsa settimana?", config_b)
for msg in result1["messages"]:
    tipo = type(msg).__name__
    if tipo == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Chiama tool: {tc['name']} con argomenti {tc['args']}")
    elif tipo == "ToolMessage":
        print(f"📥 Risultato da {msg.name}: {msg.content[:150]}")
    elif tipo == "AIMessage" and msg.content:
        print(f"🤖 Risposta finale: {msg.content}")

🔧 Chiama tool: calculate_production con argomenti {'period': 'week'}
📥 Risultato da calculate_production: Produzione settimanale:
Settimana del 02/08: 109.93 kWh
Settimana del 09/08: 273.25 kWh
Settimana del 16/08: 269.07 kWh
Settimana del 23/08: 262.87 kW
🔧 Chiama tool: load_and_clean_data con argomenti {}
📥 Risultato da load_and_clean_data: Produzione: corretti 1 valori negativi impossibili, interpolati 6 valori mancanti (sensore guasto). Consumo: corretti 0 valori negativi, interpolati 0
🤖 Risposta finale: La produzione della settimana più recente (settimana del 30 agosto) è stata di **230,19 kWh**.  

I dati sono stati prima puliti per rimuovere valori negativi e interpolare i pochi punti mancanti, così le cifre sono affidabili.  

Se ti interessa vedere l’andamento settimanale completo o confrontare produzione e consumo, fammi sapere e preparo il grafico corrispondente.


In [44]:
time.sleep(15)  # per non terminare i token al minuto

In [45]:
# Seconda domanda (stesso thread_id, quindi con memoria della prima)
result2 = invoke_agent_with_retry(agent, "E quella precedente?", config_b)
for msg in result2["messages"]:
    tipo = type(msg).__name__
    if tipo == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Chiama tool: {tc['name']} con argomenti {tc['args']}")
    elif tipo == "ToolMessage":
        print(f"📥 Risultato da {msg.name}: {msg.content[:150]}")
    elif tipo == "AIMessage" and msg.content:
        print(f"🤖 Risposta finale: {msg.content}")

🔧 Chiama tool: calculate_production con argomenti {'period': 'week'}
📥 Risultato da calculate_production: Produzione settimanale:
Settimana del 02/08: 109.93 kWh
Settimana del 09/08: 273.25 kWh
Settimana del 16/08: 269.07 kWh
Settimana del 23/08: 262.87 kW
🔧 Chiama tool: load_and_clean_data con argomenti {}
📥 Risultato da load_and_clean_data: Produzione: corretti 1 valori negativi impossibili, interpolati 6 valori mancanti (sensore guasto). Consumo: corretti 0 valori negativi, interpolati 0
🤖 Risposta finale: La produzione della settimana più recente (settimana del 30 agosto) è stata di **230,19 kWh**.  

I dati sono stati prima puliti per rimuovere valori negativi e interpolare i pochi punti mancanti, così le cifre sono affidabili.  

Se ti interessa vedere l’andamento settimanale completo o confrontare produzione e consumo, fammi sapere e preparo il grafico corrispondente.
🤖 Risposta finale: La settimana precedente (settimana del 23 agosto) ha prodotto **262,87 kWh**.  

Se vuoi conf

# STREAMLIT

In questa fase si prende tutto il codice validato nel notebook e si trasforma in un'app vera. Le parti da aggiungere rispetto al notebook sono:

- Sidebar con upload dei 2 CSV
- Chat interface (st.chat_input + st.chat_message)
- Memoria conversazionale

La logica dell'agente è stata validata in questo notebook. Per evitare duplicazione di codice tra notebook e app, il file app.py (incluso nella consegna) contiene la stessa logica di tool e agente riportata sopra, pronta per l'esecuzione con streamlit run app.py.